# 🕵️‍♂️ Forensic Intelligence Platform: Environment Check Utility
**Version:** 1.1 — Forensic Handover Suite

This notebook verifies that all external cloud services and local dependencies are correctly configured. Run this **before** triggering a full 6-phase Dagster audit.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# 1. Locate and Load Credentials
project_root = os.path.abspath("..")
os.chdir(project_root)
load_dotenv()

print(f"✅ Project Root: {project_root}")
print(f"🔑 .env Loaded: {os.path.exists('.env')}")
print(f"🔱 GCP Key Loaded: {os.path.exists('gcp-service-account.json')}")

### 🧪 1. Python Environment & Dependency Check

In [ ]:
required_packages = [
    'dagster', 'pandas', 'google.cloud.bigquery', 
    'pymongo', 'sklearn', 'xgboost', 'shap', 'textblob'
]

print("🔍 Checking Forensic Package Versions:")
for pkg in required_packages:
    try:
        m = __import__(pkg)
        print(f"  - {pkg:<20} : INSTALLED ✅")
    except ImportError:
        print(f"  - {pkg:<20} : MISSING ❌")

### 🧪 2. Cloud Database Connectivity (Bronze & Silver)

In [ ]:
from pymongo import MongoClient
from google.cloud import bigquery

# --- Test MongoDB (Bronze) ---
try:
    mongo_uri = os.getenv("MONGODB_URI")
    if mongo_uri:
        client_mongo = MongoClient(mongo_uri, serverSelectionTimeoutMS=2000)
        client_mongo.admin.command('ping')
        print("🥉 MongoDB Connectivity (Bronze Layer): SUCCESS ✅")
    else:
        print("🥉 MongoDB Connectivity: FAILED (No URI in .env) ❌")
except Exception as e:
    print(f"🥉 MongoDB Connectivity: FAILED ({e}) ❌")

# --- Test BigQuery (Silver/Gold) ---
try:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "gcp-service-account.json"
    client_bq = bigquery.Client()
    datasets = list(client_bq.list_datasets())
    print("🥇 BigQuery Connectivity (Silver/Gold): SUCCESS ✅")
except Exception as e:
    print(f"🥇 BigQuery Connectivity: FAILED ({e}) ❌")

### 🧪 3. Toolchain Binary Check

In [ ]:
import shutil

binaries = ['meltano', 'dbt']
for b in binaries:
    path = shutil.which(b)
    status = f"VALID ({path}) ✅" if path else "NOT FOUND ❌"
    print(f"⚙️ {b.upper()} Binary Check: {status}")